In [7]:
import pandas as pd
import numpy as np
from datetime import datetime

In [3]:
# Loading Cleaned Data

orders_completed = pd.read_csv(r'C:\Users\Ben Ten\OneDrive\Desktop\Projects\Final\Marketing\data\clean_orders.csv')
customers = pd.read_csv(r'C:\Users\Ben Ten\OneDrive\Desktop\Projects\Final\Marketing\data\clean_customers.csv')
responses = pd.read_csv(r'C:\Users\Ben Ten\OneDrive\Desktop\Projects\Final\Marketing\data\clean_responses.csv')

In [8]:
# Reference point for recency

SNAPSHOT_DATE = datetime(2025, 1, 1)

In [9]:
orders_completed.head()

,order_id,customer_id,order_date,amount,category,channel,status
0,5153,2698,2024-11-09,472.13,Sports,Google Search,Completed
1,5154,4151,2022-10-21,243.56,Beauty,Email,Completed
2,5156,4479,2023-07-21,186.72,Apparel,SMS,Completed
3,5159,1554,2023-02-15,1109.18,Home & Kitchen,Google Search,Completed
4,5160,2468,2022-12-27,309.73,Home & Kitchen,Organic,Completed


In [13]:
rfm = orders_completed.groupby('customer_id').agg(
    last_order_date = ('order_date', 'max'),
    frequency = ('order_id', 'count'),
    monetary = ('amount', 'sum')
).reset_index()

In [ ]:
rfm

,customer_id,last_order_date,frequency,monetary
0,1002,2022-12-06,2,614.95
1,1003,2022-08-25,1,475.38
2,1004,2024-07-06,2,1067.46
3,1005,2024-05-29,3,962.47
4,1007,2024-03-01,2,771.54
...,...,...,...,...
4527,5996,2024-11-28,2,1238.67
4528,5997,2024-05-23,6,1977.37
4529,5998,2023-12-21,3,555.18
4530,5999,2024-08-20,5,1130.15


In [16]:
rfm['last_order_date'] = pd.to_datetime(rfm['last_order_date'])

In [17]:
rfm['recency'] = (SNAPSHOT_DATE - rfm['last_order_date']).dt.days

In [18]:
rfm.head()

,customer_id,last_order_date,frequency,monetary,recency
0,1002,2022-12-06,2,614.95,757
1,1003,2022-08-25,1,475.38,860
2,1004,2024-07-06,2,1067.46,179
3,1005,2024-05-29,3,962.47,217
4,1007,2024-03-01,2,771.54,306


In [20]:
# RFM Scoring

rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5])

In [21]:
rfm.head()

,customer_id,last_order_date,frequency,monetary,recency,R_score,F_score,M_score
0,1002,2022-12-06,2,614.95,757,1,2,3
1,1003,2022-08-25,1,475.38,860,1,1,2
2,1004,2024-07-06,2,1067.46,179,4,2,4
3,1005,2024-05-29,3,962.47,217,3,3,4
4,1007,2024-03-01,2,771.54,306,3,2,3


In [22]:
rfm.info()

<class 'pandas.DataFrame'>
RangeIndex: 4532 entries, 0 to 4531
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   customer_id      4532 non-null   int64         
 1   last_order_date  4532 non-null   datetime64[us]
 2   frequency        4532 non-null   int64         
 3   monetary         4532 non-null   float64       
 4   recency          4532 non-null   int64         
 5   R_score          4532 non-null   category      
 6   F_score          4532 non-null   category      
 7   M_score          4532 non-null   category      
dtypes: category(3), datetime64[us](1), float64(1), int64(3)
memory usage: 190.6 KB


In [23]:
rfm[['R_score', 'F_score', 'M_score']] = rfm[['R_score', 'F_score', 'M_score']].astype(int)

In [24]:
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

In [25]:
rfm.head()

,customer_id,last_order_date,frequency,monetary,recency,R_score,F_score,M_score,RFM_score
0,1002,2022-12-06,2,614.95,757,1,2,3,6
1,1003,2022-08-25,1,475.38,860,1,1,2,4
2,1004,2024-07-06,2,1067.46,179,4,2,4,10
3,1005,2024-05-29,3,962.47,217,3,3,4,10
4,1007,2024-03-01,2,771.54,306,3,2,3,8


In [27]:
# RFM Segment Labels

def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f >= 2 and m >= 3:
        return 'Potential Loyalists'
    elif r == 2 and f >= 2:
        return 'At-Risk'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'Cannot Lose Them'
    elif r == 1 and f == 1:
        return 'Lost'
    else:
        return 'Hibernating'

In [29]:
rfm['rfm_segment'] = rfm.apply(assign_segment, axis=1)

In [30]:
rfm.head()

,customer_id,last_order_date,frequency,monetary,recency,R_score,F_score,M_score,RFM_score,rfm_segment
0,1002,2022-12-06,2,614.95,757,1,2,3,6,Hibernating
1,1003,2022-08-25,1,475.38,860,1,1,2,4,Lost
2,1004,2024-07-06,2,1067.46,179,4,2,4,10,New Customers
3,1005,2024-05-29,3,962.47,217,3,3,4,10,Loyal Customers
4,1007,2024-03-01,2,771.54,306,3,2,3,8,Potential Loyalists


In [31]:
rfm['rfm_segment'].value_counts()

rfm_segment
Loyal Customers        1204
champions               769
Hibernating             766
At-Risk                 702
New Customers           446
Lost                    405
Cannot Lose Them        164
Potential Loyalists      76
Name: count, dtype: int64

In [ ]:
'''
rfm.to_csv('rfm_scores.csv', index=False)
'''